In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
# Loading the dataset
data = pd.read_csv("Churn_Modelling.csv")
print(data.shape)
data.head()

(10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Preprocessing the data
# Droping the irrelevant features
data = data.drop(columns=["RowNumber", "CustomerId", "Surname"])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# Getting the information about the dataset
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 966.1 KB


In [5]:
# Encoding Categorical Variables
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [6]:
from sklearn.preprocessing import OneHotEncoder

# OneHot Encoding the Geography variable
ohe_geo = OneHotEncoder(sparse_output=False)
geo_encoder = ohe_geo.fit_transform(data[["Geography"]])
print(geo_encoder)

[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 ...
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]


In [7]:
# Getting the feature names
print(ohe_geo.get_feature_names_out(["Geography"]))

['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [8]:
# Creating the dataframe from the previous datas
geo_encoded_df = pd.DataFrame(
    geo_encoder,
    columns=ohe_geo.get_feature_names_out(["Geography"])
)
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [9]:
# Combining the new dataframe to the main dataframe
data = pd.concat(
    [data.drop("Geography", axis=1), geo_encoded_df],
    axis=1
)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [10]:
# Saving the encoders and the scaler
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(ohe_geo, file)

In [11]:
# Dividing the dataset into dependent and independent features
X = data.drop("Exited", axis=1)
y = data["Exited"]

# Splitting the dataset into training and testing datasets
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print
print(X_train.shape, X_test.shape)

(8000, 12) (2000, 12)


In [12]:
# Saving the scaler file
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

# ANN Implementation

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [14]:
from tensorflow.keras.layers import Dropout, BatchNormalization

# Building our ANN Model
model = Sequential(
    [
        # Hidden Layer 1
        Dense(
            units=64,
            activation="relu",
            input_shape=(X_train.shape[1], )
        ),
        BatchNormalization(),
        Dropout(0.3),
        
        # Hidden Layer 2
        Dense(
            units=32,
            activation="relu"
        ),
        BatchNormalization(),
        Dropout(0.2),
        
        # Output Layer
        Dense(
            units=1,
            activation="sigmoid"
        )
    ]
)

In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 batch_normalization (Batch  (None, 64)                256       
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 batch_normalization_1 (Bat  (None, 32)                128       
 chNormalization)                                                
                                                                 
 dropout_1 (Dropout)         (None, 32)                0

In [16]:
# Optimizer
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
loss = tf.keras.losses.BinaryCrossentropy()

In [17]:
# Compiling the model
model.compile(
    optimizer=opt,
    loss=loss,
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

In [18]:
# Setting up the tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [19]:
# Setting up early stoping
early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [20]:
# Training the model
history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=100,
    batch_size=32,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 4s 7ms/step - loss: 0.5630 - accuracy: 0.7270 - auc: 0.6953 - val_loss: 0.3985 - val_accuracy: 0.8395 - val_auc: 0.8134
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 0.4392 - accuracy: 0.8136 - auc: 0.7548 - val_loss: 0.3581 - val_accuracy: 0.8570 - val_auc: 0.8387
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 0.4107 - accuracy: 0.8240 - auc: 0.7924 - val_loss: 0.3495 - val_accuracy: 0.8610 - val_auc: 0.8449
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 0.4000 - accuracy: 0.8284 - auc: 0.8031 - val_loss: 0.3485 - val_accuracy: 0.8625 - val_auc: 0.8462
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3930 - accuracy: 0.8376 - auc: 0.8112 - val_loss: 0.3451 - val_accuracy: 0.8585 - val_auc: 0.8504
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3829 - accuracy: 0.8379 - auc: 0.8220 - val_lo

In [21]:
model.save("model.h5")

c:\Users\91743\Documents\ANN Classification\ann_venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [22]:
# Loading the Tensorboard Extension
%load_ext tensorboard

In [23]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 31628), started 19:02:17 ago. (Use '!kill 31628' to kill it.)

In [24]:
!kill 31628

kill: 31628: No such process
